<a href="https://colab.research.google.com/github/nananana25/MSI140/blob/main/MIS140A3_Praba_200000072.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!pip install bertopic
!pip install sentence_transformers
!pip install keybert

  Using cached keybert-0.9.0-py3-none-any.whl.metadata (15 kB)
Using cached keybert-0.9.0-py3-none-any.whl (41 kB)


In [9]:
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

import re
from collections import Counter
from nltk.corpus import stopwords
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from textblob import TextBlob
import nltk
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from keybert import KeyBERT
from nltk.util import ngrams

# NLTK resource downloads
nltk.download('vader_lexicon')
nltk.download('stopwords')

# Load stopwords
stop_words = set(stopwords.words('english'))

pd.set_option('display.max_colwidth', None)

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [11]:
url = '/content/NovoBank.csv'
df = pd.read_csv(url)

In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10127 entries, 0 to 10126
Data columns (total 17 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   customer_id                        10117 non-null  float64
 1   sex                                10127 non-null  object 
 2   age                                10117 non-null  float64
 3   marital_status                     10117 non-null  object 
 4   number_of_dependents               10127 non-null  int64  
 5   annual_income                      10127 non-null  object 
 6   account_type                       10117 non-null  object 
 7   credit_limit                       10127 non-null  float64
 8   number_of_accounts                 10127 non-null  int64  
 9   months_since_opening               10127 non-null  int64  
 10  outstanding_balance                10127 non-null  int64  
 11  utilisation_ratio                  10117 non-null  flo

In [18]:
df

,customer_id,sex,age,marital_status,number_of_dependents,annual_income,account_type,credit_limit,number_of_accounts,months_since_opening,outstanding_balance,utilisation_ratio,total_amount_of_transactions,total_number_of_transactions,number_of_contacts_over_12_months,months_inactive_over_12_months,status
0,1.431131e+09,M,50.0,Married,1,$110K and Over,Silver,7522.0,3,40,1057,0.141,2533,62,2,3,Active
1,1.427610e+09,M,41.0,Single,3,$70K - $90K,Silver,9246.0,4,33,0,0.000,4611,91,3,1,Active
2,1.425868e+09,F,49.0,Single,4,Less than $50K,Silver,2636.0,1,36,1257,0.477,7136,82,3,1,Active
3,1.559760e+09,F,53.0,Unknown,2,Less than $50K,Silver,2081.0,2,37,1342,0.645,4428,80,2,3,Active
4,1.436426e+09,M,39.0,Married,3,$110K and Over,Silver,3281.0,6,36,1682,0.513,4109,69,2,3,Active
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10122,1.434595e+09,M,50.0,Single,0,$90K - $110K,Silver,3460.0,5,38,1881,0.544,1546,35,2,2,Active
10123,1.422170e+09,M,44.0,Single,4,$90K - $110K,Silver,19719.0,3,31,1303,0.066,8385,97,3,2,Active
10124,1.439313e+09,F,52.0,Married,2,$50K - $70K,Gold,19220.0,3,42,0,0.000,4642,81,2,3,Active
10125,1.427712e+09,M,41.0,Married,4,$90K - $110K,Gold,34516.0,5,31,1433,0.042,2589,57,2,2,Active


In [20]:
df.isna().sum()

,0
customer_id,10
sex,0
age,10
marital_status,10
number_of_dependents,0
annual_income,0
account_type,10
credit_limit,0
number_of_accounts,0
months_since_opening,0


In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10127 entries, 0 to 10126
Data columns (total 17 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   customer_id                        10117 non-null  float64
 1   sex                                10127 non-null  object 
 2   age                                10117 non-null  float64
 3   marital_status                     10117 non-null  object 
 4   number_of_dependents               10127 non-null  int64  
 5   annual_income                      10127 non-null  object 
 6   account_type                       10117 non-null  object 
 7   credit_limit                       10127 non-null  float64
 8   number_of_accounts                 10127 non-null  int64  
 9   months_since_opening               10127 non-null  int64  
 10  outstanding_balance                10127 non-null  int64  
 11  utilisation_ratio                  10117 non-null  flo

In [22]:
df = df.dropna()

In [23]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10077 entries, 0 to 10126
Data columns (total 17 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   customer_id                        10077 non-null  float64
 1   sex                                10077 non-null  object 
 2   age                                10077 non-null  float64
 3   marital_status                     10077 non-null  object 
 4   number_of_dependents               10077 non-null  int64  
 5   annual_income                      10077 non-null  object 
 6   account_type                       10077 non-null  object 
 7   credit_limit                       10077 non-null  float64
 8   number_of_accounts                 10077 non-null  int64  
 9   months_since_opening               10077 non-null  int64  
 10  outstanding_balance                10077 non-null  int64  
 11  utilisation_ratio                  10077 non-null  float64
